# Notebook 01 — D1 source intake, data quality and cross-file overlap

**Scope:** Nazario.csv, Nazario_5.csv, SpamAssasin.csv, Enron.csv, CEAS_08.csv, from [Zenodo 8339691](https://zenodo.org/records/8339691).

**One run:** Execute **Run All** (CPU only). The one code cell verifies published MD5 values, reuses exact-matching attached inputs, or bounded-downloads the official files; audits all parsable rows; computes within-file duplicates and normalized cross-file overlap; and saves a **safe aggregate report** to /kaggle/working/phishing_d1_evidence/.

**Privacy:** No raw email, address, subject, body, URL, or row-level text hash is printed or included in the shareable report. Downloaded data and the row-index-only private manual-review queue are placed under /kaggle/temp (when on Kaggle), outside the saved outputs directory. Avoid publishing attached private source files.

**Scientific gate:** The results are descriptive. Source MD5 is not a ground-truth or rights certificate. No automatic deletion, relabeling, training, malicious URL opening, or public redistribution. The primary phishing-vs-legitimate task remains NO-GO until labels, rights, parent/source overlap, and split criteria are independently resolved.

**Return to chat:** Only d1_aggregate_report.json, d1_aggregate_report.txt, and any error summary (no private row review CSV).

In [ ]:
# ONE-CELL END-TO-END D1 AUDIT — CPU, stdlib, fail-closed, no training
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import csv, hashlib, json, os, re, sys, tempfile, urllib.request
from datetime import datetime, timezone

RELEASE = "D1-N01-v1.0-2026-10-31"
BASE_URL = "https://zenodo.org/records/8339691/files/"
FILES = {
    "Nazario.csv":       ("4022d055bb7cf8602f30f768f652e91e", 20_000_000),
    "Nazario_5.csv":     ("45db8330ea4aabbf72f5199949ae03e5", 25_000_000),
    "SpamAssasin.csv":   ("d921e0b2333bfaa058bd38193e6b3fbd", 30_000_000),
    "Enron.csv":         ("10489e0fdcc8af6fe62135c689367d40", 75_000_000),
    "CEAS_08.csv":       ("1f0d59191eec892a709995d48cd8decb", 100_000_000),
}
KNOWN_NAZARIO_SHA256 = "b8fbc4158fbdfaff1ed98584c43d72e283c6352d7ade4d457d34c1d79488d184"
ON_KAGGLE = Path("/kaggle").is_dir()
PRIVATE = (Path("/kaggle/temp") if ON_KAGGLE else Path(tempfile.gettempdir())) / "phishing_d1_private"
PUBLIC = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd()) / "phishing_d1_evidence"
PRIVATE.mkdir(parents=True, exist_ok=True)
PUBLIC.mkdir(parents=True, exist_ok=True)
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

def fingerprints(path):
    md5, sha = hashlib.md5(), hashlib.sha256()  # MD5 only matches a published source fingerprint
    with path.open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            md5.update(part)
            sha.update(part)
    return md5.hexdigest(), sha.hexdigest()

def verified(path, filename):
    if not path.is_file():
        return None
    if path.stat().st_size > FILES[filename][1]:
        raise ValueError("Oversized candidate " + filename)
    md5, sha = fingerprints(path)
    if md5 != FILES[filename][0]:
        return None
    if filename == "Nazario.csv" and sha != KNOWN_NAZARIO_SHA256:
        raise ValueError("Nazario SHA256 differs despite MD5 match")
    return {"path": path, "sha256": sha, "md5": md5}

def locate(filename):
    # Check the previously used private workspace without modifying it.
    old = Path("/kaggle/working/AI-Generated-Phishing-Detection/data/datasets/D1_curated_emails/raw") / filename
    if old.exists():
        result = verified(old, filename)
        if result is None:
            raise ValueError("Existing private source checksum mismatch: " + filename)
        result["source_mode"] = "existing_local_verified"
        return result
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for p in sorted(input_root.rglob(filename)):
            if p.is_file():
                result = verified(p, filename)
                if result is not None:
                    result["source_mode"] = "attached_input_verified"
                    return result
    target = PRIVATE / filename
    if target.exists():
        result = verified(target, filename)
        if result is None:
            raise ValueError("Existing temporary source checksum mismatch: " + filename)
        result["source_mode"] = "temporary_file_reused"
        return result
    return None

def fetch_official(filename):
    target = PRIVATE / filename
    temporary = PRIVATE / (filename + ".partial")
    max_bytes = FILES[filename][1]
    request = urllib.request.Request(
        BASE_URL + filename + "?download=1",
        headers={"User-Agent": "AcademicIntegrityAudit/1.0"},
    )
    md5, sha = hashlib.md5(), hashlib.sha256()
    nbytes = 0
    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            size = response.headers.get("Content-Length")
            if size and int(size) > max_bytes:
                raise ValueError("Remote Content-Length too large: " + filename)
            with temporary.open("wb") as dst:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    nbytes += len(block)
                    if nbytes > max_bytes:
                        raise ValueError("Download exceeds size cap: " + filename)
                    md5.update(block)
                    sha.update(block)
                    dst.write(block)
        if md5.hexdigest() != FILES[filename][0]:
            raise ValueError("Published MD5 does not match: " + filename)
        if filename == "Nazario.csv" and sha.hexdigest() != KNOWN_NAZARIO_SHA256:
            raise ValueError("Known SHA256 does not match: " + filename)
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return {"path": target, "md5": md5.hexdigest(), "sha256": sha.hexdigest(),
            "source_mode": "official_zenodo_download"}

def digest_pair(subject, body, normalized=True):
    if normalized:
        subject = " ".join(subject.casefold().split())
        body = " ".join(body.casefold().split())
    return hashlib.sha256(json.dumps((subject, body), ensure_ascii=False).encode("utf-8")).hexdigest()

def scan_csv(path, filename):
    counts, blanks, exact, norm = Counter(), Counter(), Counter(), Counter()
    norm_labels, admin_rows, dup_rows, norm_first = defaultdict(set), set(), set(), {}
    rows, invalid_rows = 0, 0
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, strict=True)
        original_headers = reader.fieldnames or []
        if not original_headers or len(original_headers) != len(set(original_headers)):
            raise ValueError("Missing or repeated CSV headers: " + filename)
        headers = {c.strip().casefold(): c for c in original_headers}
        # Preserve only validated column names in the public report.
        public_columns = [c if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,39}", c)
                          else "<unusual_column_name>" for c in original_headers]
        label_field = headers.get("label")
        subject_field, body_field = headers.get("subject"), headers.get("body")
        if not label_field:
            raise ValueError("No label column in " + filename + "; cannot audit labels")
        if not (subject_field or body_field):
            raise ValueError("No subject/body columns in " + filename + "; no overlap audit")
        for rownum, row in enumerate(reader, start=2):
            if None in row or any(v is None for v in row.values()):
                raise ValueError("Malformed CSV row (extra/missing fields) in " + filename)
            rows += 1
            for col, value in row.items():
                if not value.strip():
                    key = col if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,39}", col) else "<other>"
                    blanks[key] += 1
            label = row[label_field].strip()
            if len(label) > 32 or not re.fullmatch(r"[A-Za-z0-9_.-]+", label or ""):
                label = "<invalid_or_unusual>"
                invalid_rows += 1
            counts[label] += 1
            subject = row[subject_field] if subject_field else ""
            body = row[body_field] if body_field else ""
            key, raw = digest_pair(subject, body), digest_pair(subject, body, normalized=False)
            exact[raw] += 1
            norm[key] += 1
            norm_labels[key].add(label)
            if key in norm_first:
                dup_rows.update((norm_first[key], rownum))
            else:
                norm_first[key] = rownum
            joined = (subject + " " + body).casefold()
            if ("don't delete this message" in joined or
                    "folder internal data" in joined):
                admin_rows.add(rownum)
    if rows == 0:
        raise ValueError("Empty CSV: " + filename)
    report = {
        "rows": rows, "columns": public_columns,
        "candidate_label_counts": dict(counts),
        "invalid_or_unusual_label_rows": invalid_rows,
        "blank_counts": dict(blanks),
        "exact_subject_body_duplicate_extra_rows": sum(v - 1 for v in exact.values() if v > 1),
        "normalized_subject_body_duplicate_extra_rows": sum(v - 1 for v in norm.values() if v > 1),
        "normalized_subject_body_conflicting_label_groups": sum(len(v) > 1 for v in norm_labels.values()),
        "heuristic_admin_candidates": len(admin_rows),
        "normalized_unique_text_keys": len(norm),
    }
    queue = [(filename, r, "admin_candidate") for r in sorted(admin_rows)]
    queue += [(filename, r, "normalized_duplicate_group") for r in sorted(dup_rows)]
    return report, norm, norm_labels, queue

def self_test():
    assert digest_pair(" HEllo  ", "A \n B") == digest_pair("hello", "a b")
    assert digest_pair("a", "b", False) != digest_pair("A", "B", False)
    with tempfile.TemporaryDirectory() as d:
        t = Path(d) / "synthetic.csv"
        t.write_text(
            'subject,body,label\nA,hello  world,1\na,HELLO world,1\nB,other,0\n',
            encoding="utf-8"
        )
        rep, keys, labels, queue = scan_csv(t, "synthetic.csv")
        assert rep["rows"] == 3 and rep["normalized_subject_body_duplicate_extra_rows"] == 1
        assert rep["exact_subject_body_duplicate_extra_rows"] == 0
        assert len(queue) == 2 and len(keys) == 2
        assert sum(len(v) > 1 for v in labels.values()) == 0
    return "PASS"

def main():
    check = self_test()
    report = {
        "release": RELEASE, "generated_utc": datetime.now(timezone.utc).isoformat(),
        "official_record": "https://zenodo.org/records/8339691",
        "synthetic_self_test": check, "files": {}, "pairwise_normalized_overlap": [],
        "status": "INCOMPLETE", "binary_training_gate": "NO_GO",
        "rights_gate": "PENDING_SOURCE_AND_UPSTREAM_LICENSE_REVIEW",
        "label_semantics_gate": "PENDING_SOURCE_GROUND_TRUTH_ADJUDICATION",
        "notes": [
            "MD5 identity is not evidence of source rights or correct labels.",
            "Subject/body hashes are in-memory only; no raw text or hashes are exported.",
            "Overlap is exact after casefold/whitespace normalization, not semantic/campaign deduplication.",
            "Administrative keyword matches need manual review; no rows are deleted.",
            "Spam and AI authorship must not be equated with phishing or legitimate labels.",
        ],
    }
    keysets, keylabels, queue = {}, {}, []
    for filename in FILES:
        entry = {}
        try:
            located = locate(filename)
            if located is None:
                print("Downloading official checksum-pinned source:", filename)
                located = fetch_official(filename)
            file_report, keys, labels, flagged = scan_csv(located["path"], filename)
            entry.update(file_report)
            entry.update({
                "source_mode": located["source_mode"],
                "file_bytes": located["path"].stat().st_size,
                "source_md5": located["md5"], "source_sha256": located["sha256"],
                "audit_status": "PASS_DESCRIPTIVE",
            })
            keysets[filename] = keys
            keylabels[filename] = labels
            queue.extend(flagged)
            print(filename, "| rows:", file_report["rows"],
                  "| labels:", file_report["candidate_label_counts"],
                  "| normalized duplicate extras:", file_report["normalized_subject_body_duplicate_extra_rows"])
        except (OSError, ValueError, UnicodeError, csv.Error, TimeoutError) as ex:
            entry["audit_status"] = "FAILED_OR_UNAVAILABLE"
            # Exception strings originate from known filenames or system errors: do not expose URL/body.
            entry["error_type"] = type(ex).__name__
            print(filename, "| FAILED:", type(ex).__name__)
        report["files"][filename] = entry
    for a, b in combinations(FILES, 2):
        if a not in keysets or b not in keysets:
            continue
        shared = keysets[a].keys() & keysets[b].keys()
        conflicts = sum(keylabels[a][k].isdisjoint(keylabels[b][k]) for k in shared)
        report["pairwise_normalized_overlap"].append({
            "left": a, "right": b, "shared_unique_text_keys": len(shared),
            "left_rows_in_shared_keys": sum(keysets[a][k] for k in shared),
            "right_rows_in_shared_keys": sum(keysets[b][k] for k in shared),
            "disjoint_candidate_label_groups": conflicts,
        })
    # PRIVATE ONLY: no subjects/bodies, addresses, URLs, or text hashes; never publish this file.
    private_queue = PRIVATE / "D1_private_manual_review_row_indices.csv"
    with private_queue.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["file", "csv_line_number", "candidate_reason"])
        writer.writerows(sorted(set(queue)))
    report["private_manual_review_candidate_entries"] = len(set(queue))
    report["files_passed"] = sum(v.get("audit_status") == "PASS_DESCRIPTIVE" for v in report["files"].values())
    report["files_expected"] = len(FILES)
    report["status"] = ("DESCRIPTIVE_AUDIT_COMPLETE_RESEARCH_GATES_PENDING"
                        if report["files_passed"] == len(FILES) else
                        "INCOMPLETE_FILES_FAILED_OR_UNAVAILABLE")
    out_json = PUBLIC / "d1_aggregate_report.json"
    out_txt = PUBLIC / "d1_aggregate_report.txt"
    out_json.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    lines = [
        "D1 NOTEBOOK 01 SUMMARY", "Release: " + RELEASE, "Status: " + report["status"],
        "Sources passed: " + str(report["files_passed"]) + "/" + str(report["files_expected"]),
        "Self-test: " + check, "Binary training gate: NO_GO",
        "Shared normalized keys (source pairs):",
    ]
    for pair in report["pairwise_normalized_overlap"]:
        lines.append("  " + pair["left"] + " <-> " + pair["right"] + ": " +
                     str(pair["shared_unique_text_keys"]))
    lines += ["Manual review candidate entries: " + str(report["private_manual_review_candidate_entries"]),
              "Rights/labels/parent-safe splits remain pending.",
              "Share only d1_aggregate_report.json or d1_aggregate_report.txt."]
    out_txt.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n" + "\n".join(lines))
    print("Shareable report:", out_json)
    if report["files_passed"] != len(FILES):
        print("STOP: One or more source audits failed. Fix input/network and rerun; no training.")
    return report

D1_REPORT = main()

## What to send back

After **Run All**, send the text from the final summary and ideally the **d1_aggregate_report.json** saved under /kaggle/working/phishing_d1_evidence/. The JSON contains aggregate source statistics only. Do **not** attach any Nazario/Enron/SpamAssasin/CEAS raw CSV, the private D1 row-index queue, or screenshots showing email bodies or live links.

**GO meaning:** The five-source descriptive audit may complete, but training, label ground truth, third-party redistribution, and parent-safe split approvals remain pending. Next we will inspect source overlap and decide the next notebook/revision from the reported evidence.